In [ ]:
# | default_exp tokenisation.bpe

# Byte-pair encoding from scratch


In [ ]:
import random
from collections import Counter

import lorem
import regex as re
from tqdm import tqdm

In [ ]:
random.seed(42)

In byte-pair encoding we build a vocabulary by iteratively merging the most frequent pair of adjacent tokens.

1. Start with individual characters as tokens.
2. Find the most frequent pair of adjacent tokens.
3. Merge that pair into a new token.
4. Repeat until we reach the desired size of our vocabulary.


We start off by creating a small corpus of text.


In [ ]:
text = lorem.text()

print(text)

Eius eius dolor tempora consectetur sed. Amet quaerat modi aliquam adipisci amet dolorem eius. Quiquia adipisci porro dolorem sit quisquam sit porro. Eius neque quaerat est velit adipisci ut. Sit modi ipsum est dolor.

Consectetur amet magnam consectetur labore labore. Est velit aliquam tempora neque porro consectetur magnam. Porro etincidunt voluptatem quisquam. Labore quaerat dolorem sit amet aliquam sed eius. Amet eius consectetur magnam est neque. Dolore labore labore dolorem sed est.

Quiquia quisquam dolore porro. Dolore neque magnam est quisquam. Eius sed ipsum voluptatem ut ut aliquam eius. Velit ipsum magnam est. Dolorem quaerat sit ipsum. Quisquam non magnam quisquam neque. Est dolor eius tempora porro. Est tempora quaerat modi quaerat magnam labore eius. Numquam non amet ut aliquam. Dolor quisquam dolore velit.


## Training


In [ ]:
tokens = text.encode("utf-8")

print(tokens)

b'Eius eius dolor tempora consectetur sed. Amet quaerat modi aliquam adipisci amet dolorem eius. Quiquia adipisci porro dolorem sit quisquam sit porro. Eius neque quaerat est velit adipisci ut. Sit modi ipsum est dolor.\n\nConsectetur amet magnam consectetur labore labore. Est velit aliquam tempora neque porro consectetur magnam. Porro etincidunt voluptatem quisquam. Labore quaerat dolorem sit amet aliquam sed eius. Amet eius consectetur magnam est neque. Dolore labore labore dolorem sed est.\n\nQuiquia quisquam dolore porro. Dolore neque magnam est quisquam. Eius sed ipsum voluptatem ut ut aliquam eius. Velit ipsum magnam est. Dolorem quaerat sit ipsum. Quisquam non magnam quisquam neque. Est dolor eius tempora porro. Est tempora quaerat modi quaerat magnam labore eius. Numquam non amet ut aliquam. Dolor quisquam dolore velit.'


In [ ]:
tokens = list(map(int, tokens))

print(tokens)
print(f"Length: {len(tokens)}")

[69, 105, 117, 115, 32, 101, 105, 117, 115, 32, 100, 111, 108, 111, 114, 32, 116, 101, 109, 112, 111, 114, 97, 32, 99, 111, 110, 115, 101, 99, 116, 101, 116, 117, 114, 32, 115, 101, 100, 46, 32, 65, 109, 101, 116, 32, 113, 117, 97, 101, 114, 97, 116, 32, 109, 111, 100, 105, 32, 97, 108, 105, 113, 117, 97, 109, 32, 97, 100, 105, 112, 105, 115, 99, 105, 32, 97, 109, 101, 116, 32, 100, 111, 108, 111, 114, 101, 109, 32, 101, 105, 117, 115, 46, 32, 81, 117, 105, 113, 117, 105, 97, 32, 97, 100, 105, 112, 105, 115, 99, 105, 32, 112, 111, 114, 114, 111, 32, 100, 111, 108, 111, 114, 101, 109, 32, 115, 105, 116, 32, 113, 117, 105, 115, 113, 117, 97, 109, 32, 115, 105, 116, 32, 112, 111, 114, 114, 111, 46, 32, 69, 105, 117, 115, 32, 110, 101, 113, 117, 101, 32, 113, 117, 97, 101, 114, 97, 116, 32, 101, 115, 116, 32, 118, 101, 108, 105, 116, 32, 97, 100, 105, 112, 105, 115, 99, 105, 32, 117, 116, 46, 32, 83, 105, 116, 32, 109, 111, 100, 105, 32, 105, 112, 115, 117, 109, 32, 101, 115, 116, 32, 100,

In [ ]:
pair_counts: dict[tuple[int, int], int] = {}
for pair in zip(tokens, tokens[1:]):
    pair_counts[pair] = pair_counts.get(pair, 0) + 1

print(pair_counts)

{(69, 105): 3, (105, 117): 10, (117, 115): 10, (115, 32): 6, (32, 101): 14, (101, 105): 7, (32, 100): 9, (100, 111): 9, (111, 108): 15, (108, 111): 13, (111, 114): 29, (114, 32): 8, (32, 116): 4, (116, 101): 11, (101, 109): 11, (109, 112): 4, (112, 111): 9, (114, 97): 10, (97, 32): 6, (32, 99): 4, (99, 111): 4, (111, 110): 7, (110, 115): 5, (115, 101): 9, (101, 99): 5, (99, 116): 5, (101, 116): 12, (116, 117): 5, (117, 114): 5, (32, 115): 8, (101, 100): 4, (100, 46): 1, (46, 32): 18, (32, 65): 2, (65, 109): 2, (109, 101): 6, (116, 32): 31, (32, 113): 12, (113, 117): 32, (117, 97): 19, (97, 101): 6, (101, 114): 6, (97, 116): 8, (32, 109): 10, (109, 111): 3, (111, 100): 3, (100, 105): 6, (105, 32): 6, (32, 97): 12, (97, 108): 5, (108, 105): 9, (105, 113): 7, (97, 109): 24, (109, 32): 26, (97, 100): 3, (105, 112): 7, (112, 105): 3, (105, 115): 10, (115, 99): 3, (99, 105): 4, (114, 101): 15, (115, 46): 4, (32, 81): 2, (81, 117): 3, (117, 105): 11, (105, 97): 2, (32, 112): 5, (114, 114): 6,

You can print the sorted pair counts like this:


In [ ]:
print(sorted(pair_counts.items(), key=lambda kv: kv[1], reverse=True))

[((113, 117), 32), ((116, 32), 31), ((111, 114), 29), ((109, 32), 26), ((97, 109), 24), ((117, 97), 19), ((46, 32), 18), ((111, 108), 15), ((114, 101), 15), ((32, 101), 14), ((108, 111), 13), ((101, 116), 12), ((32, 113), 12), ((32, 97), 12), ((101, 32), 12), ((116, 101), 11), ((101, 109), 11), ((117, 105), 11), ((105, 117), 10), ((117, 115), 10), ((114, 97), 10), ((32, 109), 10), ((105, 115), 10), ((32, 100), 9), ((100, 111), 9), ((112, 111), 9), ((115, 101), 9), ((108, 105), 9), ((105, 116), 9), ((115, 116), 9), ((114, 32), 8), ((32, 115), 8), ((97, 116), 8), ((101, 105), 7), ((111, 110), 7), ((105, 113), 7), ((105, 112), 7), ((115, 113), 7), ((32, 110), 7), ((109, 97), 7), ((97, 103), 7), ((103, 110), 7), ((110, 97), 7), ((115, 32), 6), ((97, 32), 6), ((109, 101), 6), ((97, 101), 6), ((101, 114), 6), ((100, 105), 6), ((105, 32), 6), ((114, 114), 6), ((114, 111), 6), ((101, 115), 6), ((97, 98), 6), ((98, 111), 6), ((110, 115), 5), ((101, 99), 5), ((99, 116), 5), ((116, 117), 5), ((11

Next we need to pick the pair with the highest count as our first merge.


In [ ]:
most_common_pair = max(pair_counts, key=lambda p: pair_counts[p])
print(most_common_pair)
print(f"Count: {pair_counts[most_common_pair]}")
print(f"{chr(most_common_pair[0])}{chr(most_common_pair[1])}")


(113, 117)
Count: 32
qu


And merge it into a new token.


In [ ]:
new_token = most_common_pair[0] + most_common_pair[1]
print(new_token)

230


We now update our words to use the new token.


In [ ]:
new_tokens = []
i = 0
while i < len(tokens):
    if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == most_common_pair:
        new_tokens.append(new_token)
        i += 2
    else:
        new_tokens.append(tokens[i])
        i += 1

print(new_tokens)
print(f"Length: {len(new_tokens)}")

[69, 105, 117, 115, 32, 101, 105, 117, 115, 32, 100, 111, 108, 111, 114, 32, 116, 101, 109, 112, 111, 114, 97, 32, 99, 111, 110, 115, 101, 99, 116, 101, 116, 117, 114, 32, 115, 101, 100, 46, 32, 65, 109, 101, 116, 32, 230, 97, 101, 114, 97, 116, 32, 109, 111, 100, 105, 32, 97, 108, 105, 230, 97, 109, 32, 97, 100, 105, 112, 105, 115, 99, 105, 32, 97, 109, 101, 116, 32, 100, 111, 108, 111, 114, 101, 109, 32, 101, 105, 117, 115, 46, 32, 81, 117, 105, 230, 105, 97, 32, 97, 100, 105, 112, 105, 115, 99, 105, 32, 112, 111, 114, 114, 111, 32, 100, 111, 108, 111, 114, 101, 109, 32, 115, 105, 116, 32, 230, 105, 115, 230, 97, 109, 32, 115, 105, 116, 32, 112, 111, 114, 114, 111, 46, 32, 69, 105, 117, 115, 32, 110, 101, 230, 101, 32, 230, 97, 101, 114, 97, 116, 32, 101, 115, 116, 32, 118, 101, 108, 105, 116, 32, 97, 100, 105, 112, 105, 115, 99, 105, 32, 117, 116, 46, 32, 83, 105, 116, 32, 109, 111, 100, 105, 32, 105, 112, 115, 117, 109, 32, 101, 115, 116, 32, 100, 111, 108, 111, 114, 46, 10, 10, 67

In [ ]:
assert not any(
    new_tokens[i] == most_common_pair[0] and new_tokens[i + 1] == most_common_pair[1]
    for i in range(len(new_tokens) - 1)
)

new_tokens.count(new_token)

32

We can see that the length of the token list has decreased by the number of times we merged the pair (833 -> 801). Meanwhile, our vocabulary has obviously increased by 1.


Let's wrap what we've done so far into some functions:


In [ ]:
def get_tokens(text: str) -> list[int]:
    tokens: bytes = text.encode(encoding="utf-8")
    return list(map(int, tokens))


def get_pair_counts(tokens: list[int]) -> dict[tuple[int, int], int]:
    counts: dict[tuple[int, int], int] = {}
    for pair in zip(tokens, tokens[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts


def merge_pair(tokens: list[int], pair: tuple[int, int], idx: int) -> list[int]:
    new_tokens = []
    i = 0
    while i < len(tokens):
        if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
            new_tokens.append(idx)
            i += 2
        else:
            new_tokens.append(tokens[i])
            i += 1
    return new_tokens

And we'll use them in a loop


In [ ]:
VOCAB_SIZE = 260
merges: dict[tuple[int, int], int] = {}
tokens = get_tokens(text)

for i in range(256, VOCAB_SIZE):
    pair_counts = get_pair_counts(tokens)
    most_common_pair = max(pair_counts, key=lambda p: pair_counts[p])
    tokens = merge_pair(tokens, most_common_pair, i)
    merges[most_common_pair] = i
    print(f"Merged {most_common_pair} into {i}. Sequence length: {len(tokens)}")

Merged (113, 117) into 256. Sequence length: 801
Merged (116, 32) into 257. Sequence length: 770
Merged (111, 114) into 258. Sequence length: 741
Merged (109, 32) into 259. Sequence length: 715


In [ ]:
print(merges)

{(113, 117): 256, (116, 32): 257, (111, 114): 258, (109, 32): 259}


And that is how we build up our vocabulary. The vocab put together looks like this:


In [ ]:
vocab = {i: bytes([i]) for i in range(256)}
for pair, idx in merges.items():
    vocab[idx] = vocab[pair[0]] + vocab[pair[1]]

print(vocab)

{0: b'\x00', 1: b'\x01', 2: b'\x02', 3: b'\x03', 4: b'\x04', 5: b'\x05', 6: b'\x06', 7: b'\x07', 8: b'\x08', 9: b'\t', 10: b'\n', 11: b'\x0b', 12: b'\x0c', 13: b'\r', 14: b'\x0e', 15: b'\x0f', 16: b'\x10', 17: b'\x11', 18: b'\x12', 19: b'\x13', 20: b'\x14', 21: b'\x15', 22: b'\x16', 23: b'\x17', 24: b'\x18', 25: b'\x19', 26: b'\x1a', 27: b'\x1b', 28: b'\x1c', 29: b'\x1d', 30: b'\x1e', 31: b'\x1f', 32: b' ', 33: b'!', 34: b'"', 35: b'#', 36: b'$', 37: b'%', 38: b'&', 39: b"'", 40: b'(', 41: b')', 42: b'*', 43: b'+', 44: b',', 45: b'-', 46: b'.', 47: b'/', 48: b'0', 49: b'1', 50: b'2', 51: b'3', 52: b'4', 53: b'5', 54: b'6', 55: b'7', 56: b'8', 57: b'9', 58: b':', 59: b';', 60: b'<', 61: b'=', 62: b'>', 63: b'?', 64: b'@', 65: b'A', 66: b'B', 67: b'C', 68: b'D', 69: b'E', 70: b'F', 71: b'G', 72: b'H', 73: b'I', 74: b'J', 75: b'K', 76: b'L', 77: b'M', 78: b'N', 79: b'O', 80: b'P', 81: b'Q', 82: b'R', 83: b'S', 84: b'T', 85: b'U', 86: b'V', 87: b'W', 88: b'X', 89: b'Y', 90: b'Z', 91: b'[',

## Decoding


We need a way to decode our tokens back into text.


In [ ]:
def decode(tokens: list[int]) -> str:
    token_bytes: bytes = b"".join(vocab[token] for token in tokens)
    return token_bytes.decode("utf-8", errors="replace")


print(decode(tokens))

Eius eius dolor tempora consectetur sed. Amet quaerat modi aliquam adipisci amet dolorem eius. Quiquia adipisci porro dolorem sit quisquam sit porro. Eius neque quaerat est velit adipisci ut. Sit modi ipsum est dolor.

Consectetur amet magnam consectetur labore labore. Est velit aliquam tempora neque porro consectetur magnam. Porro etincidunt voluptatem quisquam. Labore quaerat dolorem sit amet aliquam sed eius. Amet eius consectetur magnam est neque. Dolore labore labore dolorem sed est.

Quiquia quisquam dolore porro. Dolore neque magnam est quisquam. Eius sed ipsum voluptatem ut ut aliquam eius. Velit ipsum magnam est. Dolorem quaerat sit ipsum. Quisquam non magnam quisquam neque. Est dolor eius tempora porro. Est tempora quaerat modi quaerat magnam labore eius. Numquam non amet ut aliquam. Dolor quisquam dolore velit.


## Encoding


We also need a way to encode text into tokens, following the rules we built up in the training phase.


In [ ]:
def encode(text: str) -> list[int]:
    token_bytes: bytes = text.encode(encoding="utf-8")
    tokens: list[int] = list(map(int, token_bytes))
    for pair, idx in merges.items():
        tokens = merge_pair(tokens, pair, idx)
    return tokens


print(encode(text))
print(f"Length: {len(encode(text))}")

[69, 105, 117, 115, 32, 101, 105, 117, 115, 32, 100, 111, 108, 258, 32, 116, 101, 109, 112, 258, 97, 32, 99, 111, 110, 115, 101, 99, 116, 101, 116, 117, 114, 32, 115, 101, 100, 46, 32, 65, 109, 101, 257, 256, 97, 101, 114, 97, 257, 109, 111, 100, 105, 32, 97, 108, 105, 256, 97, 259, 97, 100, 105, 112, 105, 115, 99, 105, 32, 97, 109, 101, 257, 100, 111, 108, 258, 101, 259, 101, 105, 117, 115, 46, 32, 81, 117, 105, 256, 105, 97, 32, 97, 100, 105, 112, 105, 115, 99, 105, 32, 112, 258, 114, 111, 32, 100, 111, 108, 258, 101, 259, 115, 105, 257, 256, 105, 115, 256, 97, 259, 115, 105, 257, 112, 258, 114, 111, 46, 32, 69, 105, 117, 115, 32, 110, 101, 256, 101, 32, 256, 97, 101, 114, 97, 257, 101, 115, 257, 118, 101, 108, 105, 257, 97, 100, 105, 112, 105, 115, 99, 105, 32, 117, 116, 46, 32, 83, 105, 257, 109, 111, 100, 105, 32, 105, 112, 115, 117, 259, 101, 115, 257, 100, 111, 108, 258, 46, 10, 10, 67, 111, 110, 115, 101, 99, 116, 101, 116, 117, 114, 32, 97, 109, 101, 257, 109, 97, 103, 110, 97

In [ ]:
assert decode(encode(text)) == text


## As a class


In [ ]:
# | export
class NaiveBPE:
    def __init__(self, vocab_size: int):
        self.vocab: dict[int, bytes] = {i: bytes([i]) for i in range(256)}
        self.merges: dict[tuple[int, int], int] = {}
        self.vocab_size = vocab_size

    def encode(self, text: str) -> list[int]:
        token_bytes: bytes = text.encode(encoding="utf-8")
        tokens: list[int] = list(map(int, token_bytes))
        for pair, idx in self.merges.items():
            tokens = self.merge_pair(tokens, pair, idx)
        return tokens

    def decode(self, tokens: list[int]) -> str:
        token_bytes: bytes = b"".join(self.vocab[token] for token in tokens)
        return token_bytes.decode("utf-8", errors="replace")

    def train(self, text: str):
        tokens = self.get_tokens(text)
        for i in range(256, self.vocab_size):
            pair_counts = self.get_pair_counts(tokens)
            most_common_pair = max(pair_counts, key=lambda p: pair_counts[p])
            tokens = self.merge_pair(tokens, most_common_pair, i)
            self.merges[most_common_pair] = i
            self.vocab[i] = self.vocab[most_common_pair[0]] + self.vocab[most_common_pair[1]]

    def get_tokens(self, text: str) -> list[int]:
        tokens: bytes = text.encode(encoding="utf-8")
        return list(map(int, tokens))

    def get_pair_counts(self, tokens: list[int]) -> dict[tuple[int, int], int]:
        counts: dict[tuple[int, int], int] = {}
        for pair in zip(tokens, tokens[1:]):
            counts[pair] = counts.get(pair, 0) + 1
        return counts

    def merge_pair(self, tokens: list[int], pair: tuple[int, int], idx: int) -> list[int]:
        new_tokens = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
                new_tokens.append(idx)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        return new_tokens

In [ ]:
naive_bpe = NaiveBPE(vocab_size=260)

In [ ]:
naive_bpe.train(text)

### Tests


In [ ]:
# Vocab has the right size
assert len(naive_bpe.vocab) == 260, f"Expected 260, got {len(naive_bpe.vocab)}"

# Base vocab entries are single bytes
assert naive_bpe.vocab[65] == b"A"
assert naive_bpe.vocab[32] == b" "

# Merged vocab entries are bytes concatenations of their component tokens
for pair, idx in naive_bpe.merges.items():
    assert naive_bpe.vocab[idx] == naive_bpe.vocab[pair[0]] + naive_bpe.vocab[pair[1]], (
        f"vocab[{idx}] mismatch for pair {pair}"
    )

# Encoding reduces sequence length vs raw UTF-8
raw_len = len(list(map(int, text.encode("utf-8"))))
encoded = naive_bpe.encode(text)
assert len(encoded) < raw_len, "Encoding should shorten the sequence"

# Round-trip: decode(encode(text)) == text
assert naive_bpe.decode(naive_bpe.encode(text)) == text, "Round-trip failed"

# Encoding unseen text still produces a valid token sequence
unseen = "Hello world"
encoded_unseen = naive_bpe.encode(unseen)
assert isinstance(encoded_unseen, list)
assert all(isinstance(t, int) for t in encoded_unseen)
assert naive_bpe.decode(encoded_unseen) == unseen

## Avoiding merging across word boundaries


We'll use the regex from GPT-4 to split the text into words.


In [ ]:
GPT4_SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""

In [ ]:
text_chunks = re.findall(GPT4_SPLIT_PATTERN, text)

print(text_chunks)

['Eius', ' eius', ' dolor', ' tempora', ' consectetur', ' sed', '.', ' Amet', ' quaerat', ' modi', ' aliquam', ' adipisci', ' amet', ' dolorem', ' eius', '.', ' Quiquia', ' adipisci', ' porro', ' dolorem', ' sit', ' quisquam', ' sit', ' porro', '.', ' Eius', ' neque', ' quaerat', ' est', ' velit', ' adipisci', ' ut', '.', ' Sit', ' modi', ' ipsum', ' est', ' dolor', '.\n\n', 'Consectetur', ' amet', ' magnam', ' consectetur', ' labore', ' labore', '.', ' Est', ' velit', ' aliquam', ' tempora', ' neque', ' porro', ' consectetur', ' magnam', '.', ' Porro', ' etincidunt', ' voluptatem', ' quisquam', '.', ' Labore', ' quaerat', ' dolorem', ' sit', ' amet', ' aliquam', ' sed', ' eius', '.', ' Amet', ' eius', ' consectetur', ' magnam', ' est', ' neque', '.', ' Dolore', ' labore', ' labore', ' dolorem', ' sed', ' est', '.\n\n', 'Quiquia', ' quisquam', ' dolore', ' porro', '.', ' Dolore', ' neque', ' magnam', ' est', ' quisquam', '.', ' Eius', ' sed', ' ipsum', ' voluptatem', ' ut', ' ut', ' al

When we happy to tokenise across any part of the text we encoded it all at once. but now we have a list of "text chunks".


In [ ]:
tokenised_chunks = [list(ch.encode("utf-8")) for ch in text_chunks]

print(tokenised_chunks)


[[69, 105, 117, 115], [32, 101, 105, 117, 115], [32, 100, 111, 108, 111, 114], [32, 116, 101, 109, 112, 111, 114, 97], [32, 99, 111, 110, 115, 101, 99, 116, 101, 116, 117, 114], [32, 115, 101, 100], [46], [32, 65, 109, 101, 116], [32, 113, 117, 97, 101, 114, 97, 116], [32, 109, 111, 100, 105], [32, 97, 108, 105, 113, 117, 97, 109], [32, 97, 100, 105, 112, 105, 115, 99, 105], [32, 97, 109, 101, 116], [32, 100, 111, 108, 111, 114, 101, 109], [32, 101, 105, 117, 115], [46], [32, 81, 117, 105, 113, 117, 105, 97], [32, 97, 100, 105, 112, 105, 115, 99, 105], [32, 112, 111, 114, 114, 111], [32, 100, 111, 108, 111, 114, 101, 109], [32, 115, 105, 116], [32, 113, 117, 105, 115, 113, 117, 97, 109], [32, 115, 105, 116], [32, 112, 111, 114, 114, 111], [46], [32, 69, 105, 117, 115], [32, 110, 101, 113, 117, 101], [32, 113, 117, 97, 101, 114, 97, 116], [32, 101, 115, 116], [32, 118, 101, 108, 105, 116], [32, 97, 100, 105, 112, 105, 115, 99, 105], [32, 117, 116], [46], [32, 83, 105, 116], [32, 109, 11

In [ ]:
print(f"Number of chunks: {len(tokenised_chunks)}")
print(f"Number of tokens: {sum([len(tc) for tc in tokenised_chunks])}")

Number of chunks: 145
Number of tokens: 833


We can still reuse the functions we created before. But we'll add some tweaks to the training loop to handle the fact that we're now operating on a list of chunks.


In [ ]:
VOCAB_SIZE = 260
merges: dict[tuple[int, int], int] = {}
tokens = get_tokens(text)

for i in range(256, VOCAB_SIZE):
    pair_counts = {}
    for tc in tokenised_chunks:
        for pair, count in get_pair_counts(tc).items():
            pair_counts[pair] = pair_counts.get(pair, 0) + count
    most_common_pair = max(pair_counts, key=lambda p: pair_counts[p])
    tokenised_chunks = [merge_pair(tc, most_common_pair, i) for tc in tokenised_chunks]
    merges[most_common_pair] = i

In [ ]:
print(f"Merges: {merges}")
print(f"Number of chunks: {len(tokenised_chunks)}")
print(f"Number of tokens: {sum([len(tc) for tc in tokenised_chunks])}")

Merges: {(113, 117): 256, (111, 114): 257, (97, 109): 258, (111, 108): 259}
Number of chunks: 145
Number of tokens: 733


We still get a good amount of compression, even with the restriction of only merging within chunks.


In [ ]:
print(tokenised_chunks)

[[69, 105, 117, 115], [32, 101, 105, 117, 115], [32, 100, 259, 257], [32, 116, 101, 109, 112, 257, 97], [32, 99, 111, 110, 115, 101, 99, 116, 101, 116, 117, 114], [32, 115, 101, 100], [46], [32, 65, 109, 101, 116], [32, 256, 97, 101, 114, 97, 116], [32, 109, 111, 100, 105], [32, 97, 108, 105, 256, 258], [32, 97, 100, 105, 112, 105, 115, 99, 105], [32, 258, 101, 116], [32, 100, 259, 257, 101, 109], [32, 101, 105, 117, 115], [46], [32, 81, 117, 105, 256, 105, 97], [32, 97, 100, 105, 112, 105, 115, 99, 105], [32, 112, 257, 114, 111], [32, 100, 259, 257, 101, 109], [32, 115, 105, 116], [32, 256, 105, 115, 256, 258], [32, 115, 105, 116], [32, 112, 257, 114, 111], [46], [32, 69, 105, 117, 115], [32, 110, 101, 256, 101], [32, 256, 97, 101, 114, 97, 116], [32, 101, 115, 116], [32, 118, 101, 108, 105, 116], [32, 97, 100, 105, 112, 105, 115, 99, 105], [32, 117, 116], [46], [32, 83, 105, 116], [32, 109, 111, 100, 105], [32, 105, 112, 115, 117, 109], [32, 101, 115, 116], [32, 100, 259, 257], [46, 

Let's build our vocab from the merges again.


In [ ]:
vocab = {i: bytes([i]) for i in range(256)}
for pair, idx in merges.items():
    vocab[idx] = vocab[pair[0]] + vocab[pair[1]]

### Encoding with chunks


Having not merged across chunks, we also need to encode each chunk independently.


In [ ]:
encoded_tokens = []
for tc in text_chunks:
    encoded_tokens.append(encode(tc))

In [ ]:
print(encoded_tokens)

[[69, 105, 117, 115], [32, 101, 105, 117, 115], [32, 100, 259, 257], [32, 116, 101, 109, 112, 257, 97], [32, 99, 111, 110, 115, 101, 99, 116, 101, 116, 117, 114], [32, 115, 101, 100], [46], [32, 65, 109, 101, 116], [32, 256, 97, 101, 114, 97, 116], [32, 109, 111, 100, 105], [32, 97, 108, 105, 256, 258], [32, 97, 100, 105, 112, 105, 115, 99, 105], [32, 258, 101, 116], [32, 100, 259, 257, 101, 109], [32, 101, 105, 117, 115], [46], [32, 81, 117, 105, 256, 105, 97], [32, 97, 100, 105, 112, 105, 115, 99, 105], [32, 112, 257, 114, 111], [32, 100, 259, 257, 101, 109], [32, 115, 105, 116], [32, 256, 105, 115, 256, 258], [32, 115, 105, 116], [32, 112, 257, 114, 111], [46], [32, 69, 105, 117, 115], [32, 110, 101, 256, 101], [32, 256, 97, 101, 114, 97, 116], [32, 101, 115, 116], [32, 118, 101, 108, 105, 116], [32, 97, 100, 105, 112, 105, 115, 99, 105], [32, 117, 116], [46], [32, 83, 105, 116], [32, 109, 111, 100, 105], [32, 105, 112, 115, 117, 109], [32, 101, 115, 116], [32, 100, 259, 257], [46, 

### Decoding with chunks


In [ ]:
decoded_tokens = [decode(tc) for tc in encoded_tokens]
decoded_text = "".join(decoded_tokens)

print(decoded_text)

Eius eius dm t  tempt a consectetur sed. Amet quaerat modi aliquor adipisci oret dm t em eius. Quiquia adipisci pt ro dm t em sit quisquor sit pt ro. Eius neque quaerat est velit adipisci ut. Sit modi ipsum est dm t .

Consectetur oret magnor consectetur labt e labt e. Est velit aliquor tempt a neque pt ro consectetur magnor. Pt ro etincidunt vm uptatem quisquor. Labt e quaerat dm t em sit oret aliquor sed eius. Amet eius consectetur magnor est neque. Dm t e labt e labt e dm t em sed est.

Quiquia quisquor dm t e pt ro. Dm t e neque magnor est quisquor. Eius sed ipsum vm uptatem ut ut aliquor eius. Velit ipsum magnor est. Dm t em quaerat sit ipsum. Quisquor non magnor quisquor neque. Est dm t  eius tempt a pt ro. Est tempt a quaerat modi quaerat magnor labt e eius. Numquor non oret ut aliquor. Dm t  quisquor dm t e velit.


## As a class


In [ ]:
class ChunkedBPE:
    def __init__(self, vocab_size: int):
        self.vocab: dict[int, bytes] = {i: bytes([i]) for i in range(256)}
        self.merges: dict[tuple[int, int], int] = {}
        self.vocab_size = vocab_size

    def encode(self, text: str):
        pass

    def decode(self, tokens: list[int]):
        pass

    def train(self, text: str):
        pass


In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()